# 04 — Evaluación final y análisis de errores

Ejecute este cuaderno después de seleccionar la configuración usando exclusivamente train/validation en el cuaderno 03. Aquí se carga el conjunto de prueba una vez y cada artefacto produce una única pasada de predicción; no se ajustan hiperparámetros ni se elige el modelo usando test. La comparación incluye RF, SVM, CNN y la línea base. AUC-ROC se informa como indefinida (NaN con motivo) si faltan clases o no coinciden las columnas de probabilidad.

Los tamaños disponibles en el código son relativos a los píxeles de la imagen. No equivalen a diámetro físico; esta entrega solo podrá reportar error de tamaño en mm si se recopilan referencias físicas y una salida de modelo en las mismas unidades.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra el cuaderno desde la raíz del repositorio.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.dataset import IMAGE_EXTENSIONS
from src.evaluation.metrics import full_report
from src.evaluation.evaluate_cnn import build_test_generator
from src.utils.config import load_label_info

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
RESULTS_DIR = ROOT / 'experiments' / 'results' / 'comparison'
TRAINING_DETAILS = ROOT / 'experiments' / 'results' / 'training' / 'training_details.json'
required_paths = [
    PROCESSED_DIR / 'test.csv', PROCESSED_DIR / 'Xtestml.npy',
    PROCESSED_DIR / 'ytestml.npy', TRAINING_DETAILS,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        'Faltan datos/modelos de 02 o 03: ' + ', '.join(missing_paths)
    )

label_info = load_label_info(PROCESSED_DIR)
class_names = label_info['class_names']
label_map = label_info['label_map']
X_test = np.load(PROCESSED_DIR / 'Xtestml.npy')
y_test = np.load(PROCESSED_DIR / 'ytestml.npy')
test_manifest = pd.read_csv(PROCESSED_DIR / 'test.csv')

# Reproduce exactamente el orden clase/ruta usado por load_split_features.
test_image_paths = []
for quality in class_names:
    folder = PROCESSED_DIR / 'test' / quality
    image_paths = []
    for extension in IMAGE_EXTENSIONS:
        image_paths.extend(folder.glob(f'*{extension}'))
        image_paths.extend(folder.glob(f'*{extension.upper()}'))
    test_image_paths.extend(sorted(set(image_paths)))

if len(test_image_paths) != len(y_test) or len(X_test) != len(y_test):
    raise ValueError('Los arrays de test no coinciden en cantidad de ejemplos.')
required_manifest_columns = {'processed_path', 'quality', 'path', 'fruit', 'source'}
missing_columns = required_manifest_columns - set(test_manifest.columns)
if missing_columns:
    raise ValueError(f'test.csv no contiene {sorted(missing_columns)}')
manifest_by_path = {
    str(Path(row['processed_path']).resolve()): row
    for row in test_manifest.to_dict(orient='records')
}
ordered_rows = []
for image_path in test_image_paths:
    key = str(image_path.resolve())
    if key not in manifest_by_path:
        raise ValueError(f'No hay trazabilidad en test.csv para {image_path}')
    ordered_rows.append(manifest_by_path[key])
test_rows = pd.DataFrame(ordered_rows)
manifest_labels = test_rows['quality'].map(label_map).to_numpy(dtype=np.int32)
if not np.array_equal(manifest_labels, y_test):
    raise ValueError('Las etiquetas de test no coinciden con el orden de las imágenes.')
print('Test:', len(y_test), 'ejemplos | clases:', pd.Series(test_rows['quality']).value_counts().to_dict())

with TRAINING_DETAILS.open(encoding='utf-8') as file:
    training_details = json.load(file)
RESULT_SCOPE = training_details['result_scope']
print('Alcance de entrenamiento:', RESULT_SCOPE)


## Evaluación única del conjunto de prueba

Se conservan las predicciones y probabilidades para las tablas, las matrices de confusión y el análisis de errores. Las matrices muestran tanto conteos como porcentajes normalizados por clase real. `Balanced Accuracy` y Macro-F1 complementan accuracy cuando hay desbalance; Macro-Precision/Macro-Recall ponderan por igual las clases configuradas.

In [ ]:
predictions = {}
probabilities = {}

rf_path = next((path for path in (
    MODELS_DIR / 'randomforest.pkl', MODELS_DIR / 'random_forest.pkl'
) if path.is_file()), None)
if rf_path is None:
    raise FileNotFoundError(f'No se encontró Random Forest en {MODELS_DIR}.')
rf = joblib.load(rf_path)
predictions['Random Forest'] = rf.predict(X_test)
probabilities['Random Forest'] = rf.predict_proba(X_test)

svm_path = MODELS_DIR / 'svm.pkl'
scaler_path = MODELS_DIR / 'scaler.pkl'
if not svm_path.is_file() or not scaler_path.is_file():
    raise FileNotFoundError('Faltan svm.pkl o scaler.pkl; ejecute el cuaderno 03.')
svm = joblib.load(svm_path)
scaler = joblib.load(scaler_path)
X_test_scaled = scaler.transform(X_test)
predictions['SVM'] = svm.predict(X_test_scaled)
probabilities['SVM'] = svm.predict_proba(X_test_scaled)

cnn_path = next((path for path in (
    MODELS_DIR / 'cnn_best.keras', MODELS_DIR / 'cnnbest.keras'
) if path.is_file()), None)
if cnn_path is None:
    raise FileNotFoundError(f'No se encontró la CNN en {MODELS_DIR}.')
test_generator, generator_info = build_test_generator(PROCESSED_DIR, batch_size=32)
if generator_info['label_map'] != label_map:
    raise ValueError('El orden de clases de la CNN no coincide con labelinfo.json.')
generator_paths = [str(Path(path).resolve()) for path in test_generator.filepaths]
expected_paths = [str(path.resolve()) for path in test_image_paths]
if generator_paths != expected_paths:
    raise ValueError('El generador CNN y los arrays usan distinto orden de test.')
cnn = tf.keras.models.load_model(cnn_path)
probabilities['CNN'] = cnn.predict(test_generator, verbose=1)
predictions['CNN'] = probabilities['CNN'].argmax(axis=1)
if not np.array_equal(test_generator.classes, y_test):
    raise ValueError('Las etiquetas del generador CNN no coinciden con y_test.')

baseline_path = MODELS_DIR / 'baseline_most_frequent.pkl'
if not baseline_path.is_file():
    raise FileNotFoundError('Falta la línea base; ejecute el cuaderno 03.')
baseline = joblib.load(baseline_path)
predictions['Baseline'] = baseline.predict(X_test)
probabilities['Baseline'] = baseline.predict_proba(X_test)

reports = {
    name: full_report(y_test, predictions[name], probabilities[name], class_names, name)
    for name in predictions
}
metric_columns = [
    'Accuracy', 'Balanced Accuracy', 'Macro Precision', 'Macro Recall',
    'Macro F1', 'AUC-ROC', 'AUC-ROC status'
]
comparison = pd.DataFrame([
    {'Model': name, **{key: report[key] for key in metric_columns}}
    for name, report in reports.items()
])
display(comparison)
display(pd.DataFrame({
    name: {quality: report['classification_report'][quality]['support']
           for quality in class_names}
    for name, report in reports.items()
}).rename_axis('class'))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS_DIR / 'test_results.csv', index=False)

for model_name, report in reports.items():
    matrix = np.asarray(report['confusion_matrix'])
    matrix_pct = np.asarray(report['confusion_matrix_pct'])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f'{model_name} — conteos')
    sns.heatmap(matrix_pct, annot=True, fmt='.1f', cmap='Blues', vmin=0,
                vmax=100, cbar=False, xticklabels=class_names,
                yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f'{model_name} — % por clase real')
    for ax in axes:
        ax.set_xlabel('Predicción')
        ax.set_ylabel('Etiqueta real')
    fig.tight_layout()
    stem = model_name.lower().replace(' ', '_') + '_confusion'
    fig.savefig(RESULTS_DIR / f'{stem}.svg', bbox_inches='tight')
    fig.savefig(RESULTS_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')
    plt.show()

f1_by_class = pd.DataFrame({
    name: [report['classification_report'][quality]['f1-score'] for quality in class_names]
    for name, report in reports.items()
}, index=class_names)
ax = f1_by_class.plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
ax.set_title('F1 por clase en test')
ax.set_ylabel('F1')
ax.set_xlabel('Calidad')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'f1_by_class.svg', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'f1_by_class.png', dpi=300, bbox_inches='tight')
plt.show()

macro_f1 = comparison.set_index('Model')['Macro F1'].sort_values(ascending=False)
ax = macro_f1.plot(kind='bar', figsize=(8, 4), ylim=(0, 1), color='#4C78A8', rot=0)
ax.set_title('Macro-F1: modelos y línea base (test)')
ax.set_ylabel('Macro-F1')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'macro_f1_comparison.svg', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'macro_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


## Curvas de entrenamiento, errores y tamaño

La configuración se selecciona con Macro-F1 de validación del cuaderno 03. El análisis de falsos Good/falsos Bad se limita a ese modelo elegido antes de observar test. Las tablas por fruta/fuente sirven para detectar diferencias de cobertura, pero no demuestran causalidad ni generalización a poblaciones no muestreadas.

In [ ]:
cnn_history = training_details['models']['CNN']['validation'].get('history', {})
if cnn_history:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for key, label in (('loss', 'train'), ('val_loss', 'validation')):
        if key in cnn_history:
            axes[0].plot(cnn_history[key], label=label)
    for key, label in (('accuracy', 'train'), ('val_accuracy', 'validation')):
        if key in cnn_history:
            axes[1].plot(cnn_history[key], label=label)
    axes[0].set_title('CNN — pérdida')
    axes[1].set_title('CNN — exactitud')
    for ax in axes:
        ax.set_xlabel('Época')
        ax.legend()
        ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'cnn_training_curves.svg', bbox_inches='tight')
    fig.savefig(RESULTS_DIR / 'cnn_training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No hay historial CNN guardado; vuelva a ejecutar el entrenamiento del cuaderno 03.')

candidate_models = ('Random Forest', 'SVM', 'CNN')
selected_model = max(
    candidate_models,
    key=lambda name: training_details['models'][name]['validation']['macro_f1'],
)
error_table = test_rows.copy()
error_table['true_quality'] = [class_names[int(label)] for label in y_test]
error_table['predicted_quality'] = [
    class_names[int(label)] for label in predictions[selected_model]
]
error_table['correct'] = error_table['true_quality'].eq(error_table['predicted_quality'])
print('Modelo elegido con validación:', selected_model)
error_summary = (error_table.groupby(['true_quality', 'fruit', 'source'], dropna=False)
                .agg(examples=('correct', 'size'), accuracy=('correct', 'mean'))
                .sort_values('examples', ascending=False))
display(error_summary)
error_summary.to_csv(RESULTS_DIR / 'error_analysis_by_quality_fruit_source.csv')

false_good = error_table[
    error_table['predicted_quality'].eq('Good')
    & ~error_table['true_quality'].eq('Good')
]
false_bad = error_table[
    error_table['predicted_quality'].eq('Bad')
    & ~error_table['true_quality'].eq('Bad')
]
print('Falsos Good:', len(false_good), '| Falsos Bad:', len(false_bad))
display(false_good[['path', 'fruit', 'source', 'true_quality', 'predicted_quality']].head(10))
display(false_bad[['path', 'fruit', 'source', 'true_quality', 'predicted_quality']].head(10))
error_table.to_csv(RESULTS_DIR / 'test_predictions_selected_model.csv', index=False)

def save_error_examples(frame, title, output_stem):
    sample = frame.head(8)
    if sample.empty:
        print(f'Sin ejemplos para: {title}')
        return
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, (_, row) in zip(axes.flat, sample.iterrows()):
        ax.imshow(plt.imread(row['processed_path']))
        ax.set_title(f"real={row['true_quality']}\npred={row['predicted_quality']}", fontsize=9)
        ax.axis('off')
    for ax in axes.flat[len(sample):]:
        ax.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f'{output_stem}.svg', bbox_inches='tight')
    fig.savefig(RESULTS_DIR / f'{output_stem}.png', dpi=300, bbox_inches='tight')
    plt.show()

save_error_examples(false_good, 'Falsos Good del modelo elegido', 'false_good_examples')
save_error_examples(false_bad, 'Falsos Bad del modelo elegido', 'false_bad_examples')

if 'diameter_mm' in test_rows.columns:
    measured = pd.to_numeric(test_rows['diameter_mm'], errors='coerce').notna().sum()
    print(f'Medidas físicas de diámetro disponibles en test: {measured}/{len(test_rows)}')
print('El sistema actual estima diámetro normalizado en píxeles; no se reporta MAE físico en mm sin una salida calibrada en mm.')

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'No se puede serializar {type(value).__name__}')

detailed_results = {
    'result_scope': RESULT_SCOPE,
    'selected_model_by_validation_macro_f1': selected_model,
    'test_reports': reports,
}
with (RESULTS_DIR / 'detailed_test_results.json').open('w', encoding='utf-8') as file:
    json.dump(detailed_results, file, indent=2, default=json_default, allow_nan=True)
print('Tablas y figuras guardadas en:', RESULTS_DIR)


Interprete resultados solo después de comprobar el soporte por clase, la matriz de confusión y las curvas. Si Regular tiene soporte cero, Macro-F1 de tres clases no representa una evaluación real de esa clase y AUC-ROC permanece indefinida; el informe debe reportar el experimento como piloto Good/Bad. Para validar tamaño físico se requiere medir cada producto con regla/calibre, guardar `diameter_mm`, calibrar una salida del sistema en mm y calcular MAE/RMSE sobre productos de prueba independientes.